In [2]:
import pandas as pd

In [3]:
from pathlib import Path

DATA_PATH = Path("../data/raw")

In [4]:
csv_files = list(DATA_PATH.glob("*.csv"))

for file in csv_files:
    print(file.name)

olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [5]:
customers = pd.read_csv(DATA_PATH / "olist_customers_dataset.csv")
orders = pd.read_csv(DATA_PATH / "olist_orders_dataset.csv")
order_items = pd.read_csv(DATA_PATH / "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_PATH / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(DATA_PATH / "olist_order_reviews_dataset.csv")
products = pd.read_csv(DATA_PATH / "olist_products_dataset.csv")
sellers = pd.read_csv(DATA_PATH / "olist_sellers_dataset.csv")

In [6]:
datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")

customers: 99,441 rows × 5 columns
orders: 99,441 rows × 8 columns
order_items: 112,650 rows × 7 columns
payments: 103,886 rows × 5 columns
reviews: 99,224 rows × 7 columns
products: 32,951 rows × 9 columns
sellers: 3,095 rows × 4 columns


In [7]:
for name, df in datasets.items():
    print(f"\n{'=' * 60}")
    print(f"{name.upper()}")
    print(f"{'=' * 60}")
    print(df.columns.tolist())


CUSTOMERS
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

ORDERS
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

ORDER_ITEMS
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

PAYMENTS
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

REVIEWS
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

PRODUCTS
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

SELLERS
['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']


In [8]:
for name, df in datasets.items():
    print(f"\n{'=' * 60}")
    print(f"{name.upper()}")
    print(f"{'=' * 60}")
    
    print("\nData types:")
    print(df.dtypes)
    
    print("\nMissing values:")
    print(df.isna().sum())
    
    print("\nDuplicate rows:")
    print(df.duplicated().sum())


CUSTOMERS

Data types:
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

Missing values:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Duplicate rows:
0

ORDERS

Data types:
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

Missing values:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_da

In [9]:
quality_summary = []

for name, df in datasets.items():
    quality_summary.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_values": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum())
    })

quality_df = pd.DataFrame(quality_summary)

quality_df

,dataset,rows,columns,missing_values,duplicate_rows
0,customers,99441,5,0,0
1,orders,99441,8,4908,0
2,order_items,112650,7,0,0
3,payments,103886,5,0,0
4,reviews,99224,7,145903,0
5,products,32951,9,2448,0
6,sellers,3095,4,0,0


In [10]:
for name, df in datasets.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]

    if len(missing) > 0:
        print(f"\n{'=' * 60}")
        print(f"{name.upper()}")
        print(f"{'=' * 60}")
        print(missing)


ORDERS
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

REVIEWS
review_comment_title      87656
review_comment_message    58247
dtype: int64

PRODUCTS
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


In [11]:
checks = {
    "customers.customer_id": customers["customer_id"].nunique(),
    "customers.rows": len(customers),
    "orders.order_id": orders["order_id"].nunique(),
    "orders.rows": len(orders),
    "products.product_id": products["product_id"].nunique(),
    "products.rows": len(products),
    "sellers.seller_id": sellers["seller_id"].nunique(),
}

for check, value in checks.items():
    print(f"{check}: {value:,}")

customers.customer_id: 99,441
customers.rows: 99,441
orders.order_id: 99,441
orders.rows: 99,441
products.product_id: 32,951
products.rows: 32,951
sellers.seller_id: 3,095


In [12]:
items_per_order = (
    order_items
    .groupby("order_id")
    .size()
)

print("Orders with 1 item:", (items_per_order == 1).sum())
print("Orders with multiple items:", (items_per_order > 1).sum())
print("Maximum items in one order:", items_per_order.max())

Orders with 1 item: 88863
Orders with multiple items: 9803
Maximum items in one order: 21


In [13]:
payments_per_order = (
    payments
    .groupby("order_id")
    .size()
)

print("Orders with 1 payment:", (payments_per_order == 1).sum())
print("Orders with multiple payments:", (payments_per_order > 1).sum())
print("Maximum payments for one order:", payments_per_order.max())

Orders with 1 payment: 96479
Orders with multiple payments: 2961
Maximum payments for one order: 29


In [14]:
orders_per_customer = (
    orders
    .groupby("customer_id")
    .size()
)

print("Customers with 1 order:", (orders_per_customer == 1).sum())
print("Customers with multiple orders:", (orders_per_customer > 1).sum())
print("Maximum orders by one customer:", orders_per_customer.max())

Customers with 1 order: 99441
Customers with multiple orders: 0
Maximum orders by one customer: 1


In [15]:
orders_with_customer = orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)

orders_per_unique_customer = (
    orders_with_customer
    .groupby("customer_unique_id")
    .size()
)

print("Unique customers with 1 order:",
      (orders_per_unique_customer == 1).sum())

print("Unique customers with multiple orders:",
      (orders_per_unique_customer > 1).sum())

print("Maximum orders by one customer:",
      orders_per_unique_customer.max())

Unique customers with 1 order: 93099
Unique customers with multiple orders: 2997
Maximum orders by one customer: 17


In [16]:
missing_customers = orders[
    ~orders["customer_id"].isin(customers["customer_id"])
]

print("Orders without matching customer:", len(missing_customers))

Orders without matching customer: 0


In [17]:
missing_orders = order_items[
    ~order_items["order_id"].isin(orders["order_id"])
]

print("Order items without matching order:", len(missing_orders))

Order items without matching order: 0


In [18]:
missing_products = order_items[
    ~order_items["product_id"].isin(products["product_id"])
]

print("Order items without matching product:", len(missing_products))

Order items without matching product: 0


In [19]:
missing_sellers = order_items[
    ~order_items["seller_id"].isin(sellers["seller_id"])
]

print("Order items without matching seller:", len(missing_sellers))

Order items without matching seller: 0


In [20]:
missing_payment_orders = payments[
    ~payments["order_id"].isin(orders["order_id"])
]

print("Payments without matching order:", len(missing_payment_orders))

Payments without matching order: 0


In [21]:
missing_review_orders = reviews[
    ~reviews["order_id"].isin(orders["order_id"])
]

print("Reviews without matching order:", len(missing_review_orders))

Reviews without matching order: 0


In [22]:
print("Order item sequence statistics:")
print(order_items["order_item_id"].describe())

print("\nPrice statistics:")
print(order_items["price"].describe())

print("\nFreight value statistics:")
print(order_items["freight_value"].describe())

Order item sequence statistics:
count    112650.000000
mean          1.197834
std           0.705124
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          21.000000
Name: order_item_id, dtype: float64

Price statistics:
count    112650.000000
mean        120.653739
std         183.633928
min           0.850000
25%          39.900000
50%          74.990000
75%         134.900000
max        6735.000000
Name: price, dtype: float64

Freight value statistics:
count    112650.000000
mean         19.990320
std          15.806405
min           0.000000
25%          13.080000
50%          16.260000
75%          21.150000
max         409.680000
Name: freight_value, dtype: float64


In [23]:
print("Negative prices:", (order_items["price"] < 0).sum())
print("Zero prices:", (order_items["price"] == 0).sum())

print("Negative freight values:", (order_items["freight_value"] < 0).sum())
print("Zero freight values:", (order_items["freight_value"] == 0).sum())

Negative prices: 0
Zero prices: 0
Negative freight values: 0
Zero freight values: 383


In [24]:
print("Review score statistics:")
print(reviews["review_score"].describe())

print("\nInvalid review scores:",
      ((reviews["review_score"] < 1) |
       (reviews["review_score"] > 5)).sum())

print("\nMissing review scores:",
      reviews["review_score"].isna().sum())

Review score statistics:
count    99224.000000
mean         4.086421
std          1.347579
min          1.000000
25%          4.000000
50%          5.000000
75%          5.000000
max          5.000000
Name: review_score, dtype: float64

Invalid review scores: 0

Missing review scores: 0


In [25]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

orders[date_columns].dtypes

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

In [26]:
print("Invalid purchase dates:",
      orders["order_purchase_timestamp"].isna().sum())

print("Invalid estimated delivery dates:",
      orders["order_estimated_delivery_date"].isna().sum())

print("Invalid approved dates:",
      orders["order_approved_at"].isna().sum())

print("Invalid carrier delivery dates:",
      orders["order_delivered_carrier_date"].isna().sum())

print("Invalid customer delivery dates:",
      orders["order_delivered_customer_date"].isna().sum())

Invalid purchase dates: 0
Invalid estimated delivery dates: 0
Invalid approved dates: 160
Invalid carrier delivery dates: 1783
Invalid customer delivery dates: 2965


In [27]:
timeline_checks = {
    "approved_before_purchase": (
        orders["order_approved_at"] < orders["order_purchase_timestamp"]
    ).sum(),

    "carrier_before_purchase": (
        orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"]
    ).sum(),

    "customer_delivery_before_purchase": (
        orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"]
    ).sum(),

    "customer_delivery_before_carrier": (
        orders["order_delivered_customer_date"]
        < orders["order_delivered_carrier_date"]
    ).sum(),
}

for check, count in timeline_checks.items():
    print(f"{check}: {count}")

approved_before_purchase: 0
carrier_before_purchase: 166
customer_delivery_before_purchase: 0
customer_delivery_before_carrier: 23


In [28]:
timeline_errors = orders[
    (
        orders["order_delivered_carrier_date"]
        < orders["order_purchase_timestamp"]
    )
    |
    (
        orders["order_delivered_customer_date"]
        < orders["order_delivered_carrier_date"]
    )
]

timeline_errors[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].head(20)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
615,b9afddbdcfadc9a87b41a83271c3e888,delivered,2018-08-16 13:50:48,2018-08-16 14:05:13,2018-08-16 13:27:00,2018-08-24 14:58:37,2018-09-04
1111,ad133696906f6a78826daa0911b7daec,delivered,2018-06-15 15:41:22,2018-06-15 16:19:23,2018-06-15 14:52:00,2018-06-22 18:09:37,2018-07-18
1329,74e033208dc13a7b8127eb8e73d09b76,delivered,2018-05-02 10:48:44,2018-05-02 11:13:45,2018-05-02 09:49:00,2018-05-07 23:06:36,2018-05-29
1372,a6b58794fd2ba533359a76c08df576e3,delivered,2018-05-14 15:18:23,2018-05-14 15:33:35,2018-05-14 13:46:00,2018-05-19 19:33:32,2018-06-08
1864,5792e0b1c8c8a2bf53af468c9a422c88,delivered,2018-07-26 13:25:14,2018-07-26 13:35:14,2018-07-26 12:42:00,2018-07-30 14:45:02,2018-08-09
2760,c3eb293fd154223498b6551a728203e8,delivered,2018-07-19 14:06:04,2018-07-19 14:22:51,2018-07-19 13:49:00,2018-07-24 19:35:36,2018-08-06
3473,b0c2a7d04b165525254254a728c50a4e,delivered,2018-06-07 13:28:30,2018-06-07 13:57:22,2018-06-07 13:22:00,2018-06-21 17:36:43,2018-07-04
3661,2033a4586b5bec3229ebc1675a8ae092,delivered,2018-06-12 10:10:25,2018-06-12 10:39:59,2018-06-12 10:09:00,2018-06-19 14:08:27,2018-07-19
4114,08adcddad19d3acf37d1fa01cb9ded1e,delivered,2018-06-27 11:16:44,2018-06-27 11:30:56,2018-06-27 10:57:00,2018-06-29 17:39:53,2018-07-18
4159,dee6298ce7d1fb2645141ef9972157aa,shipped,2018-04-30 14:06:12,2018-04-30 14:15:24,2018-04-30 12:59:00,NaT,2018-05-28


In [29]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [30]:
orders["order_status"].value_counts(normalize=True).mul(100).round(2)

order_status
delivered      97.02
shipped         1.11
canceled        0.63
unavailable     0.61
invoiced        0.32
processing      0.30
created         0.01
approved        0.00
Name: proportion, dtype: float64

In [32]:
status_delivery_check = (
    orders
    .groupby("order_status")
    .agg(
        total_orders=("order_id", "count"),
        missing_carrier_date=("order_delivered_carrier_date", lambda x: x.isna().sum()),
        missing_customer_delivery=("order_delivered_customer_date", lambda x: x.isna().sum())
    )
)

status_delivery_check

,total_orders,missing_carrier_date,missing_customer_delivery
order_status,,,
approved,2,2,2
canceled,625,550,619
created,5,5,5
delivered,96478,2,8
invoiced,314,314,314
processing,301,301,301
shipped,1107,0,1107
unavailable,609,609,609


In [33]:
delivered_missing_dates = orders[
    (orders["order_status"] == "delivered") &
    (
        orders["order_delivered_customer_date"].isna() |
        orders["order_delivered_carrier_date"].isna()
    )
]

delivered_missing_dates[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
]

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
73222,2aa91108853cecb43c84a5dc5b277475,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaT,2017-11-20 19:44:47,2017-11-14
79263,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


In [34]:
delivered_missing_dates[
    [
        "order_id",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
].assign(
    missing_carrier=lambda x: x["order_delivered_carrier_date"].isna(),
    missing_customer_delivery=lambda x: x["order_delivered_customer_date"].isna()
)

,order_id,order_delivered_carrier_date,order_delivered_customer_date,missing_carrier,missing_customer_delivery
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,2017-11-30 18:12:23,NaT,False,True
20618,f5dd62b788049ad9fc0526e3ad11a097,2018-06-25 08:05:00,NaT,False,True
43834,2ebdfc4f15f23b91474edf87475f108e,2018-07-03 13:57:00,NaT,False,True
73222,2aa91108853cecb43c84a5dc5b277475,NaT,2017-11-20 19:44:47,True,False
79263,e69f75a717d64fc5ecdfae42b2e8e086,2018-07-03 13:57:00,NaT,False,True
82868,0d3268bad9b086af767785e3f0fc0133,2018-07-03 09:28:00,NaT,False,True
92643,2d858f451373b04fb5c984a1cc2defaf,NaT,NaT,True,True
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,2018-06-12 14:10:00,NaT,False,True
98038,20edc82cf5400ce95e1afacc25798b31,2018-07-03 19:26:00,NaT,False,True


In [35]:
delivery_analysis = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_purchase_timestamp"].notna()) &
    (orders["order_delivered_customer_date"].notna())
].copy()

print("Total delivered orders:", (orders["order_status"] == "delivered").sum())
print("Orders available for delivery analysis:", len(delivery_analysis))
print("Excluded due to missing delivery date:",
      (orders["order_status"] == "delivered").sum() - len(delivery_analysis))

Total delivered orders: 96478
Orders available for delivery analysis: 96470
Excluded due to missing delivery date: 8


In [36]:
delivery_analysis["delivery_days"] = (
    delivery_analysis["order_delivered_customer_date"]
    - delivery_analysis["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

delivery_analysis["delivery_days"].describe()

count    96470.000000
mean        12.558217
std          9.546156
min          0.533414
25%          6.766204
50%         10.217477
75%         15.720182
max        209.628611
Name: delivery_days, dtype: float64

In [37]:
long_deliveries = delivery_analysis[
    delivery_analysis["delivery_days"] > 60
][
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_days"
    ]
].sort_values("delivery_days", ascending=False)

print("Orders taking more than 60 days:", len(long_deliveries))

long_deliveries.head(20)

Orders taking more than 60 days: 306


,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_days
19590,ca07593549f1816d26a572e06dc1eab6,delivered,2017-02-21 23:31:27,2017-09-19 14:36:39,2017-03-22,209.628611
55619,1b3190b2dfa9d789e1f14c05b647a14a,delivered,2018-02-23 14:57:35,2018-09-19 23:24:07,2018-03-15,208.351759
61610,440d0d17af552815d15a9e41abe49359,delivered,2017-03-07 23:59:51,2017-09-19 15:12:50,2017-04-07,195.634016
70307,2fb597c2f772eca01b1f5c561bf6cc7b,delivered,2017-03-08 18:09:02,2017-09-19 14:33:17,2017-04-17,194.850174
89130,285ab9426d6982034523a855f55a885e,delivered,2017-03-08 22:47:40,2017-09-19 14:00:04,2017-04-06,194.633611
38509,0f4519c5f1c541ddec9f21b3bddd533a,delivered,2017-03-09 13:26:57,2017-09-19 14:38:21,2017-04-11,194.049583
11399,47b40429ed8cce3aee9199792275433f,delivered,2018-01-03 09:44:01,2018-07-13 20:51:31,2018-01-19,191.463542
81401,2fe324febf907e3ea3f2aa9650869fa5,delivered,2017-03-13 20:17:10,2017-09-19 17:00:07,2017-04-05,189.863160
54480,2d7561026d542c8dbd8f0daeadf67a43,delivered,2017-03-15 11:24:27,2017-09-19 14:38:18,2017-04-13,188.134618
68769,c27815f7e3dd0b926b58552628481575,delivered,2017-03-15 23:23:17,2017-09-19 17:14:25,2017-04-10,187.743843


In [38]:
delivery_analysis["delivery_delay_days"] = (
    delivery_analysis["order_delivered_customer_date"]
    - pd.to_datetime(delivery_analysis["order_estimated_delivery_date"])
).dt.total_seconds() / (24 * 60 * 60)

delivery_analysis["delivery_delay_days"].describe()

count    96470.000000
mean       -11.178126
std         10.184354
min       -146.016123
25%        -16.244065
50%        -11.948102
75%         -6.389815
max        188.975081
Name: delivery_delay_days, dtype: float64

In [39]:
delivery_analysis["delivery_performance"] = delivery_analysis[
    "delivery_delay_days"
].apply(
    lambda x: "Late" if x > 0
    else "On Time" if x == 0
    else "Early"
)

delivery_analysis["delivery_performance"].value_counts()

delivery_performance
Early    88644
Late      7826
Name: count, dtype: int64

In [40]:
delivery_analysis["delivery_performance"].value_counts(
    normalize=True
).mul(100).round(2)

delivery_performance
Early    91.89
Late      8.11
Name: proportion, dtype: float64

In [41]:
delivery_reviews = delivery_analysis.merge(
    reviews[["order_id", "review_score"]],
    on="order_id",
    how="left"
)

print("Orders with review scores:",
      delivery_reviews["review_score"].notna().sum())

print("Orders without review scores:",
      delivery_reviews["review_score"].isna().sum())

Orders with review scores: 96353
Orders without review scores: 646


In [42]:
delivery_reviews.groupby("delivery_performance")["review_score"].agg(
    ["count", "mean", "median"]
).round(2)

,count,mean,median
delivery_performance,,,
Early,88653,4.29,5.0
Late,7700,2.57,2.0


In [43]:
late_orders = delivery_analysis[
    delivery_analysis["delivery_performance"] == "Late"
]

late_orders["delivery_delay_days"].describe()

count    7826.000000
mean        9.551776
std        13.952540
min         0.002500
25%         1.867179
50%         5.806481
75%        11.820978
max       188.975081
Name: delivery_delay_days, dtype: float64

In [44]:
delivery_customer = delivery_analysis.merge(
    customers[["customer_id", "customer_state"]],
    on="customer_id",
    how="left"
)

print("Orders with customer state:",
      delivery_customer["customer_state"].notna().sum())

print("Orders without customer state:",
      delivery_customer["customer_state"].isna().sum())

Orders with customer state: 96470
Orders without customer state: 0


In [45]:
state_delivery = (
    delivery_customer
    .groupby("customer_state")
    .agg(
        total_orders=("order_id", "count"),
        late_orders=("delivery_performance", lambda x: (x == "Late").sum())
    )
)

state_delivery["late_rate"] = (
    state_delivery["late_orders"]
    / state_delivery["total_orders"]
    * 100
).round(2)

state_delivery.sort_values("late_rate", ascending=False)

,total_orders,late_orders,late_rate
customer_state,,,
AL,397,95,23.93
MA,717,141,19.67
PI,476,76,15.97
CE,1279,196,15.32
SE,335,51,15.22
BA,3256,457,14.04
RJ,12350,1664,13.47
TO,274,35,12.77
PA,946,117,12.37


In [46]:
state_late_delay = (
    delivery_customer[
        delivery_customer["delivery_performance"] == "Late"
    ]
    .groupby("customer_state")
    .agg(
        late_orders=("order_id", "count"),
        avg_delay_days=("delivery_delay_days", "mean"),
        median_delay_days=("delivery_delay_days", "median")
    )
    .round(2)
)

state_late_delay.sort_values("avg_delay_days", ascending=False)

,late_orders,avg_delay_days,median_delay_days
customer_state,,,
AP,3,48.86,1.54
RR,5,37.09,10.07
AM,6,20.90,2.72
AC,3,19.03,24.07
SE,51,16.89,9.80
CE,196,14.33,9.83
RN,51,13.15,7.79
RJ,1664,12.85,8.78
PI,76,12.30,5.85


In [47]:
state_late_delay[
    state_late_delay["late_orders"] >= 100
].sort_values(
    "avg_delay_days",
    ascending=False
)

,late_orders,avg_delay_days,median_delay_days
customer_state,,,
CE,196,14.33,9.83
RJ,1664,12.85,8.78
PA,117,12.30,8.35
PE,172,11.34,7.95
BA,457,11.10,6.76
ES,244,10.60,6.17
MA,141,9.97,7.03
GO,160,9.79,5.70
RS,382,9.41,6.85


In [48]:
state_satisfaction = (
    delivery_reviews
    .merge(
        customers[["customer_id", "customer_state"]],
        on="customer_id",
        how="left"
    )
    .groupby(["customer_state", "delivery_performance"])["review_score"]
    .agg(["count", "mean"])
    .round(2)
)

state_satisfaction

count  mean
customer_state delivery_performance             
AC             Early                    77  4.18
               Late                      3  1.67
AL             Early                   305  4.31
               Late                     93  2.31
AM             Early                   139  4.28
               Late                      6  2.83
AP             Early                    63  4.22
               Late                      3  4.67
BA             Early                  2799  4.15
               Late                    447  2.57
CE             Early                  1081  4.25
               Late                    195  2.19
DF             Early                  1940  4.26
               Late                    149  2.43
ES             Early                  1743  4.26
               Late                    235  2.74
GO             Early                  1806  4.24
               Late                    157  2.54
MA             Early                   579  4.18
               Late                    137  2.40
MG             Early                 10729  4.28
               Late                    625  2.67
MS             Early                   628  4.37
               Late                     82  2.63
MT             Early                   824  4.25
               Late                     58  2.66
PA             Early                   828  4.15
               Late                    111  2.13
PB             Early                   458  4.27
               Late                     55  2.47
PE             Early                  1422  4.30
               Late                    168  2.28
PI             Early                   397  4.29
               Late                     75  2.44
PR             Early                  4676  4.31
               Late                    243  2.91
RJ             Early                 10648  4.24
               Late                   1633  2.13
RN             Early                   423  4.40
               Late                     50  2.04
RO             Early                   235  4.21
               Late                      7  2.57
RR             Early                    36  4.19
               Late                      5  1.80
RS             Early                  4981  4.31
               Late                    382  2.51
SC             Early                  3192  4.29
               Late                    341  2.61
SE             Early                   284  4.25
               Late                     50  1.94
SP             Early                 38122  4.33
               Late                   2355  2.91
TO             Early                   238  4.29
               Late                     35  3.20

In [49]:
state_rating_impact = (
    state_satisfaction
    .reset_index()
    .pivot(
        index="customer_state",
        columns="delivery_performance",
        values="mean"
    )
)

state_rating_impact["rating_gap"] = (
    state_rating_impact["Late"]
    - state_rating_impact["Early"]
).round(2)

state_rating_impact.sort_values("rating_gap")

delivery_performance,Early,Late,rating_gap
customer_state,,,
AC,4.18,1.67,-2.51
RR,4.19,1.80,-2.39
RN,4.40,2.04,-2.36
SE,4.25,1.94,-2.31
RJ,4.24,2.13,-2.11
CE,4.25,2.19,-2.06
PE,4.30,2.28,-2.02
PA,4.15,2.13,-2.02
AL,4.31,2.31,-2.00


In [50]:
products["product_category_name"].value_counts(dropna=False).head(20)

product_category_name
cama_mesa_banho                      3029
esporte_lazer                        2867
moveis_decoracao                     2657
beleza_saude                         2444
utilidades_domesticas                2335
automotivo                           1900
informatica_acessorios               1639
brinquedos                           1411
relogios_presentes                   1329
telefonia                            1134
bebes                                 919
perfumaria                            868
fashion_bolsas_e_acessorios           849
papelaria                             849
cool_stuff                            789
ferramentas_jardim                    753
pet_shop                              719
NaN                                   610
eletronicos                           517
construcao_ferramentas_construcao     400
Name: count, dtype: int64

In [51]:
print("Missing product categories:",
      products["product_category_name"].isna().sum())

print("\nTotal unique categories:",
      products["product_category_name"].nunique())

print("\nUnique categories including missing:",
      products["product_category_name"].nunique(dropna=False))

Missing product categories: 610

Total unique categories: 73

Unique categories including missing: 74


In [52]:
sales_products = order_items.merge(
    products[["product_id", "product_category_name"]],
    on="product_id",
    how="left"
)

print("Order items:", len(order_items))
print("After product merge:", len(sales_products))

print(
    "Order items without product category:",
    sales_products["product_category_name"].isna().sum()
)

Order items: 112650
After product merge: 112650
Order items without product category: 1603


In [53]:
category_sales = (
    sales_products
    .assign(
        product_category_name=lambda df:
            df["product_category_name"].fillna("Unknown")
    )
    .groupby("product_category_name")
    .agg(
        items_sold=("order_item_id", "count"),
        revenue=("price", "sum"),
        avg_price=("price", "mean"),
        freight_value=("freight_value", "sum")
    )
    .sort_values("revenue", ascending=False)
)

category_sales = category_sales.round(2)

category_sales.head(20)

,items_sold,revenue,avg_price,freight_value
product_category_name,,,,
beleza_saude,9670,1258681.34,130.16,182566.73
relogios_presentes,5991,1205005.68,201.14,100535.93
cama_mesa_banho,11115,1036988.68,93.30,204693.04
esporte_lazer,8641,988048.97,114.34,168607.51
informatica_acessorios,7827,911954.32,116.51,147318.08
moveis_decoracao,8334,729762.49,87.56,172749.30
cool_stuff,3796,635290.85,167.36,84039.10
utilidades_domesticas,6964,632248.66,90.79,146149.11
automotivo,4235,592720.11,139.96,92664.21


In [54]:
total_revenue = category_sales["revenue"].sum()

top_5_revenue = category_sales.head(5)["revenue"].sum()

top_10_revenue = category_sales.head(10)["revenue"].sum()

print("Total product revenue:", round(total_revenue, 2))
print("Top 5 revenue:", round(top_5_revenue, 2))
print("Top 5 revenue share:", round(top_5_revenue / total_revenue * 100, 2), "%")
print("Top 10 revenue:", round(top_10_revenue, 2))
print("Top 10 revenue share:", round(top_10_revenue / total_revenue * 100, 2), "%")

Total product revenue: 13591643.7
Top 5 revenue: 5400678.99
Top 5 revenue share: 39.74 %
Top 10 revenue: 8475957.56
Top 10 revenue share: 62.36 %


In [55]:
category_performance = category_sales.copy()

category_performance["revenue_share"] = (
    category_performance["revenue"]
    / category_performance["revenue"].sum()
    * 100
)

print("TOP 10 BY ITEMS SOLD")
display(
    category_performance
    .sort_values("items_sold", ascending=False)
    .head(10)
    .round(2)
)

print("\nTOP 10 BY AVERAGE PRICE")
display(
    category_performance
    .sort_values("avg_price", ascending=False)
    .head(10)
    .round(2)
)

TOP 10 BY ITEMS SOLD


,items_sold,revenue,avg_price,freight_value,revenue_share
product_category_name,,,,,
cama_mesa_banho,11115,1036988.68,93.30,204693.04,7.63
beleza_saude,9670,1258681.34,130.16,182566.73,9.26
esporte_lazer,8641,988048.97,114.34,168607.51,7.27
moveis_decoracao,8334,729762.49,87.56,172749.30,5.37
informatica_acessorios,7827,911954.32,116.51,147318.08,6.71
utilidades_domesticas,6964,632248.66,90.79,146149.11,4.65
relogios_presentes,5991,1205005.68,201.14,100535.93,8.87
telefonia,4545,323667.53,71.21,71215.79,2.38
ferramentas_jardim,4347,485256.46,111.63,98962.75,3.57



TOP 10 BY AVERAGE PRICE


,items_sold,revenue,avg_price,freight_value,revenue_share
product_category_name,,,,,
pcs,203,222963.13,1098.34,9836.30,1.64
portateis_casa_forno_e_cafe,76,47445.71,624.29,2747.86,0.35
eletrodomesticos_2,238,113317.74,476.12,10600.18,0.83
agro_industria_e_comercio,212,72530.47,342.12,5843.60,0.53
instrumentos_musicais,680,191498.88,281.62,18638.49,1.41
eletroportateis,679,190648.58,280.78,16020.25,1.40
portateis_cozinha_e_preparadores_de_alimentos,15,3968.53,264.57,309.76,0.03
telefonia_fixa,264,59583.00,225.69,4637.81,0.44
construcao_ferramentas_seguranca,194,40544.52,208.99,3919.10,0.30


In [56]:
category_performance["freight_rate"] = (
    category_performance["freight_value"]
    / category_performance["revenue"]
    * 100
)

print("TOP 15 CATEGORIES BY FREIGHT RATE")

display(
    category_performance[
        ["items_sold", "revenue", "freight_value", "freight_rate"]
    ]
    .sort_values("freight_rate", ascending=False)
    .head(15)
    .round(2)
)

TOP 15 CATEGORIES BY FREIGHT RATE


,items_sold,revenue,freight_value,freight_rate
product_category_name,,,,
casa_conforto_2,30,760.27,410.31,53.97
flores,33,1110.04,488.87,44.04
moveis_colchao_e_estofado,38,4368.08,1630.46,37.33
artigos_de_natal,153,8800.82,3229.30,36.69
fraldas_higiene,39,1567.59,573.68,36.60
cds_dvds_musicais,14,730.00,224.99,30.82
sinalizacao_e_seguranca,199,21509.23,6507.82,30.26
alimentos_bebidas,278,15179.48,4507.99,29.70
eletronicos,2767,160246.74,46578.32,29.07


In [57]:
high_volume_freight = (
    category_performance[
        category_performance["items_sold"] >= 500
    ]
    [["items_sold", "revenue", "freight_value", "freight_rate"]]
    .sort_values("freight_rate", ascending=False)
)

high_volume_freight.round(2)

,items_sold,revenue,freight_value,freight_rate
product_category_name,,,,
eletronicos,2767,160246.74,46578.32,29.07
moveis_sala,503,68916.56,17968.17,26.07
moveis_escritorio,1691,273960.70,68571.95,25.03
alimentos,510,29393.41,7271.03,24.74
moveis_decoracao,8334,729762.49,172749.30,23.67
utilidades_domesticas,6964,632248.66,146149.11,23.12
telefonia,4545,323667.53,71215.79,22.00
malas_acessorios,1092,140429.98,30445.23,21.68
fashion_bolsas_e_acessorios,2031,152823.54,31450.00,20.58


In [58]:
order_items_with_sellers = order_items.merge(
    sellers[["seller_id", "seller_zip_code_prefix", "seller_city", "seller_state"]],
    on="seller_id",
    how="left"
)

print("Order items:", len(order_items))
print("After seller merge:", len(order_items_with_sellers))

print(
    "Order items without seller:",
    order_items_with_sellers["seller_id"].isna().sum()
)

Order items: 112650
After seller merge: 112650
Order items without seller: 0


In [59]:
seller_performance = (
    order_items_with_sellers
    .groupby("seller_id")
    .agg(
        items_sold=("order_item_id", "count"),
        revenue=("price", "sum"),
        avg_price=("price", "mean"),
        freight_value=("freight_value", "sum")
    )
    .sort_values("revenue", ascending=False)
)

seller_performance.head(15)

,items_sold,revenue,avg_price,freight_value
seller_id,,,,
4869f7a5dfa277a7dca6462dcf3b52b2,1156,229472.63,198.505735,20168.07
53243585a1d6dc2643021fd1853d8905,410,222776.05,543.356220,13080.63
4a3ca9315b744ce9f8e9374361493884,1987,200472.92,100.892260,35067.04
fa1c13f2614d7b5c4749cbc52fecda94,586,194042.03,331.129744,10042.70
7c67e1448b00f6e969d365cea6b010ab,1364,187923.89,137.774113,51612.55
7e93a43ef30c4f03f38b393420bc753a,340,176431.87,518.917265,6322.18
da8622b14eb17ae2831f4ac5b9dab84a,1551,160236.57,103.311779,24955.75
7a67c85e85bb2ce8582c35f2203ad736,1171,141745.53,121.046567,20902.85
1025f0e2d44d7041d6cf58b6550e0bfa,1428,138968.55,97.316912,33892.14


In [60]:
total_seller_revenue = seller_performance["revenue"].sum()

seller_performance["revenue_share"] = (
    seller_performance["revenue"]
    / total_seller_revenue
    * 100
)

top_5_seller_revenue = seller_performance.head(5)["revenue"].sum()
top_10_seller_revenue = seller_performance.head(10)["revenue"].sum()

print("Total seller revenue:", round(total_seller_revenue, 2))

print("Top 5 seller revenue:", round(top_5_seller_revenue, 2))
print(
    "Top 5 seller revenue share:",
    round(top_5_seller_revenue / total_seller_revenue * 100, 2),
    "%"
)

print("Top 10 seller revenue:", round(top_10_seller_revenue, 2))
print(
    "Top 10 seller revenue share:",
    round(top_10_seller_revenue / total_seller_revenue * 100, 2),
    "%"
)

Total seller revenue: 13591643.7
Top 5 seller revenue: 1034687.52
Top 5 seller revenue share: 7.61 %
Top 10 seller revenue: 1787241.74
Top 10 seller revenue share: 13.15 %


In [61]:
seller_order_counts = (
    order_items_with_sellers
    .groupby("seller_id")
    .size()
)

print("Total sellers:", sellers["seller_id"].nunique())

print(
    "Sellers with orders:",
    seller_order_counts.shape[0]
)

print(
    "Sellers with 1 item:",
    (seller_order_counts == 1).sum()
)

print(
    "Sellers with multiple items:",
    (seller_order_counts > 1).sum()
)

print(
    "Maximum items sold by one seller:",
    seller_order_counts.max()
)

Total sellers: 3095
Sellers with orders: 3095
Sellers with 1 item: 509
Sellers with multiple items: 2586
Maximum items sold by one seller: 2033


In [62]:
seller_state_performance = (
    order_items_with_sellers
    .groupby("seller_state")
    .agg(
        sellers=("seller_id", "nunique"),
        items_sold=("order_item_id", "count"),
        revenue=("price", "sum"),
        avg_price=("price", "mean"),
        freight_value=("freight_value", "sum")
    )
    .sort_values("revenue", ascending=False)
)

seller_state_performance.head(15)

,sellers,items_sold,revenue,avg_price,freight_value
seller_state,,,,,
SP,1849,80342,8753396.21,108.951684,1482487.67
PR,349,8671,1261887.21,145.529606,197013.52
MG,244,8827,1011564.74,114.598928,212595.06
RJ,171,4818,843984.22,175.173147,93829.90
SC,190,4075,632426.07,155.196582,106547.06
RS,129,2199,378559.54,172.150769,57243.09
BA,19,643,285561.56,444.108180,19700.68
DF,30,899,97749.48,108.731346,18494.06
PE,9,448,91493.85,204.227344,12392.46
